# 07 · Transformer (PatchTST) frente a LSTM apilada — documentación y reporte

**Pregunta que responde.** Plan de mejora 5.2 y matriz de riesgos del Prof. Marcos Alania: *«En 2026, los Transformers como PatchTST dominan. ¿Por qué usar una LSTM de 1997?»*. Se pidió comparar el Recall de eventos severos y la latencia de inferencia para justificar si la LSTM sigue siendo la mejor opción para el despliegue.

**Qué hace este notebook.** Documenta el experimento, muestra el script empleado y **genera tablas y figuras desde el registro, sin reentrenar**.

> ⚠️ **No reentrenar desde aquí.** El experimento se ejecutó con `scripts/correr_patchtst.py`.

| Elemento | Valor |
|---|---|
| Configuración común | D' · S4 + hora + ROTI · ventana 180 min · horizonte 20 min · WFL α = 1.5, β = 10 (notebooks 05 y 06) |
| LSTM apilada | 64/64 unidades, dropout 0.3, Dense 64 → 20 (elegida en el torneo de arquitecturas) |
| PatchTST (Nie et al., ICLR 2023) | Parches de 16 min con paso 8 (21 tokens por canal), independencia de canal, RevIN, d_model 64, 4 cabezas, 2 capas, d_ff 128, dropout 0.2 |
| Evaluación | Validación, 3 semillas (42, 123, 7), umbral de alerta calibrado por modelo |
| Latencia | 200 pronósticos de una ventana, en CPU (despliegue típico) y GPU (RTX 4080 SUPER) |

## 1. Pasos seguidos

| # | Paso | Herramienta |
|---|---|---|
| 1 | Implementar PatchTST en Keras con el mismo contrato que la LSTM (misma entrada, salida y pérdida) | `src/s4/models/patchtst.py`, probado en `tests/test_modelos.py` |
| 2 | Entrenar ambos modelos con la configuración elegida, 3 semillas y umbral calibrado | `scripts/correr_patchtst.py` |
| 3 | Medir latencia de inferencia de una ventana en CPU y GPU | `train.medir_latencia` (dentro del mismo script) |
| 4 | Probar la hipótesis de que la normalización por ventana (RevIN) oculta el nivel absoluto de S4: PatchTST idéntico con `revin=False` | `scripts/correr_patchtst_sin_revin.py` |
| 5 | Reporte | **este notebook** |

> El notebook 07 original (configuración `E_completo`, 60 min, β = 100) quedó obsoleto tras la ablación y la sensibilidad α–β; sus corridas, si existen, están en el grupo `transformer` del registro y no se usan.

## 2. Script empleado: `scripts/correr_patchtst.py`

```bash
nohup python scripts/correr_patchtst.py > experiments/log_patchtst.txt 2>&1 &
```

```python
cfg = variante(cargar_config("config/ablacion_2025_2026.yaml"),
               **{"ventanas.lookback": 180, "perdida.alpha": 1.5, "perdida.beta": 10.0})
datos = preparar_datos(cfg)
PATCH = {"tipo": "patchtst", "patch_len": 16, "stride": 8, "d_model": 64,
         "n_heads": 4, "n_capas": 2, "d_ff": 128, "dropout": 0.2}
MODELOS = [("LSTM apilada", cfg), ("PatchTST", variante(cfg, modelo=PATCH))]
latencias = []
for nombre, c in MODELOS:
    filas, modelos, parts = correr_variante(c, datos, ft.CONJUNTOS["D_solo_ROTI"], nombre, "transformer_D",
                                            barrido_umbral=True, guardar_modelo=True)
    modelo = modelos[0][1]
    for disp in ["/CPU:0", "/GPU:0"]:
        latencias.append({"modelo": nombre, **medir_latencia(modelo, parts["val"], n=200, dispositivo=disp)})
pd.DataFrame(latencias).to_csv("reports/tables/tabla_latencia_patchtst_lstm.csv", index=False)
```

## 2b. Script complementario: `scripts/correr_patchtst_sin_revin.py`

**Hipótesis.** RevIN centra y escala cada ventana con su propia media y desviación: el modelo ve la *forma* de la serie pero no su *nivel absoluto*, que es justamente lo que decide si S4 se acerca al umbral de 0.6. Se entrena el mismo PatchTST sin RevIN (mismos 230 676 parámetros, misma configuración, 3 semillas) y se registra en el mismo grupo.

```python
PATCH = {"tipo": "patchtst", "patch_len": 16, "stride": 8, "d_model": 64,
         "n_heads": 4, "n_capas": 2, "d_ff": 128, "dropout": 0.2, "revin": False}
filas, modelos, parts = correr_variante(variante(cfg, modelo=PATCH), datos, ft.CONJUNTOS["D_solo_ROTI"],
                                        "PatchTST sin RevIN", "transformer_D", barrido_umbral=True, guardar_modelo=True)
```

## 3. Reporte a partir del registro (no entrena)

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, ruta
from s4.registry import leer, resumir_semillas

cfg = cargar_config("../config/ablacion_2025_2026.yaml")
REG = ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv"
FIG, TAB = ruta(cfg, "reportes") / "figures", ruta(cfg, "reportes") / "tables"
r = leer(REG, grupo="transformer_D", particion="val")
t = resumir_semillas(r, por=("modelo",), metricas=("HSS", "HSS_opt", "umbral_opt", "Precision_opt", "POD_inicio_opt", "RMSE_evento", "RMSE_global"))
lat = pd.read_csv(TAB / "tabla_latencia_patchtst_lstm.csv")
t.to_csv(TAB / "tabla_patchtst_vs_lstm.csv", index=False)
display(t.round(3)); display(lat.round(2))

In [ ]:
orden = [o for o in ["LSTM apilada", "PatchTST", "PatchTST sin RevIN"] if o in set(t["modelo"])]
col = {"LSTM apilada": "#0000AF", "PatchTST": "#0099FF", "PatchTST sin RevIN": "#8FB8E8"}
t = t.set_index("modelo").reindex(orden); latc = lat[lat["dispositivo"] == "/CPU:0"].set_index("modelo").reindex(orden)
fig, axes = plt.subplots(1, 4, figsize=(19, 4.6))
paneles = [("HSS_opt", "HSS con umbral calibrado\n(mayor es mejor)", True),
           ("Precision_opt", "Precisión de la alerta\n(mayor es mejor)", True),
           ("RMSE_evento", "RMSE en eventos\n(menor es mejor)", True)]
for ax, (m, tit, err) in zip(axes, paneles):
    ax.bar(orden, t[f"{m}_mean"], yerr=t[f"{m}_std"], capsize=5, color=[col[o] for o in orden])
    tope = (t[f"{m}_mean"] + t[f"{m}_std"]).max()
    for i, (v, e) in enumerate(zip(t[f"{m}_mean"], t[f"{m}_std"])):
        ax.text(i, v + e + tope * 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=11)
    ax.set_ylim(0, tope * 1.18)
    ax.set_title(tit, fontsize=11); ax.spines[["top", "right"]].set_visible(False); ax.tick_params(axis="x", labelsize=9)
ax = axes[3]
ax.bar(orden, latc["lat_ms_mediana"], color=[col[o] for o in orden])
for i, (v, n) in enumerate(zip(latc["lat_ms_mediana"], latc["parametros"])):
    ax.text(i, v, f"{v:.1f} ms\n{n/1000:.0f} k parám.", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, latc["lat_ms_mediana"].max() * 1.35)
ax.set_title("Latencia por pronóstico en CPU\n(menor es mejor)", fontsize=11); ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("PatchTST frente a LSTM apilada · misma configuración, 3 semillas (validación)")
plt.tight_layout(); plt.savefig(FIG / "fig_patchtst_vs_lstm.png", dpi=200); plt.show()

## 4. Resultados obtenidos (validación, 3 semillas)

| Métrica | LSTM apilada | PatchTST | PatchTST sin RevIN |
|---|---|---|---|
| HSS con umbral calibrado | **0.661 ± 0.009** | 0.600 ± 0.021 | 0.594 ± 0.052 |
| HSS con umbral fijo 0.6 | **0.630 ± 0.014** | 0.592 ± 0.015 | 0.551 ± 0.114 |
| Umbral óptimo | 0.55 (las 3 semillas) | 0.55 (las 3 semillas) | 0.55 (las 3 semillas) |
| Precisión de la alerta* | **65 % ± 3 %** | 54 % ± 3 % | 55 % ± 2 % |
| Inicios anticipados* | 61 % ± 5 % | 62 % ± 3 % | 59 % ± 10 % |
| RMSE en eventos | **0.188 ± 0.012** | 0.234 ± 0.002 | 0.233 ± 0.043 |
| RMSE global | **0.059 ± 0.003** | 0.070 ± 0.003 | 0.064 ± 0.003 |
| Parámetros | **56 660** | 230 676 | 230 676 |
| Latencia en CPU (mediana / p95) | **7.9 / 9.7 ms** | 16.9 / 20.8 ms | 16.8 / 35.1 ms |
| Latencia en GPU (mediana / p95) | **6.0 / 6.5 ms** | 12.8 / 13.3 ms | 12.5 / 13.0 ms |

\* Con el umbral calibrado.

## 5. Conclusiones

1. **La LSTM apilada supera al PatchTST en todas las métricas de calidad**: +0.06 de HSS calibrado (el doble de la variación observada entre entrenamientos idénticos, ≈ 0.03), 11 puntos más de precisión y menor error en eventos (0.188 frente a 0.234) y global (0.059 frente a 0.070).
2. **Ambos anticipan la misma proporción de inicios** (61–62 %): la ventaja de la LSTM está en emitir menos falsas alarmas.
3. **La LSTM es más eficiente:** 4 veces menos parámetros y la mitad de latencia (7.9 ms frente a 16.9 ms en CPU). Ambas latencias son irrelevantes frente a un pronóstico que se actualiza cada minuto, por lo que la latencia no decide la elección, pero refuerza la de la LSTM para un despliegue *Edge*.
4. **La LSTM es más estable entre semillas** (HSS ± 0.009 frente a ± 0.021).
5. **RevIN no explica la diferencia.** Sin RevIN, el PatchTST obtiene la misma habilidad de alerta (0.594 frente a 0.600) y la misma precisión (55 % frente a 54 %), pero se vuelve mucho más inestable entre semillas (HSS con umbral fijo ± 0.114, inicios anticipados ± 10 %). La hipótesis de que la normalización por ventana ocultaba el nivel absoluto de S4 se **descarta**: RevIN no perjudica la detección y, en cambio, estabiliza el entrenamiento. Las explicaciones que quedan en pie son la ventana corta (21 parches por canal), la independencia de canal y la falta de optimización del Transformer.
6. **Respuesta a la pregunta del especialista:** en las mismas condiciones, la LSTM apilada obtuvo mejor habilidad de alerta, con menos parámetros, menor latencia y menor variabilidad. Una explicación plausible es que, con ventanas de 180 min (21 parches por canal), el Transformer dispone de una secuencia corta para aprovechar su capacidad, que suele destacar con historias largas; la hipótesis alternativa de la normalización por ventana se probó y se descartó.

**Limitación.** El PatchTST se evaluó con una configuración estándar (la del artículo original, adaptada a 180 min), sin la búsqueda de hiperparámetros que sí tuvo la LSTM (torneo de arquitecturas y sensibilidad α–β). La conclusión es que, con configuración estándar, PatchTST no supera a la LSTM; una optimización exhaustiva del Transformer queda como trabajo futuro.